In [ ]:
import os
import geopandas as gpd
import pandas as pd

# 设置文件夹路径
folder_path = r"E:\chuli\Point_to_csv"

# 获取文件夹中所有的shp文件
shp_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.shp')]

print(f"找到 {len(shp_files)} 个shp文件:")
for shp_file in shp_files:
    print(f"  - {shp_file}")

# 遍历每个shp文件并转换为csv
for shp_file in shp_files:
    try:
        # 构建完整的文件路径
        shp_path = os.path.join(folder_path, shp_file)
        
        # 读取shp文件
        gdf = gpd.read_file(shp_path)
        
        # 检查几何类型是否为点
        if gdf.geometry.type.iloc[0] == 'Point':
            # 提取经纬度坐标
            gdf['longitude'] = gdf.geometry.x
            gdf['latitude'] = gdf.geometry.y
        else:
            # 如果不是点类型，提取边界框的中心点作为近似坐标
            gdf['longitude'] = gdf.geometry.centroid.x
            gdf['latitude'] = gdf.geometry.centroid.y
        
        # 移除geometry列，只保留属性和坐标
        df = pd.DataFrame(gdf.drop(columns='geometry'))
        
        # 构建输出csv文件名
        csv_filename = os.path.splitext(shp_file)[0] + '.csv'
        csv_path = os.path.join(folder_path, csv_filename)
        
        # 保存为csv文件
        df.to_csv(csv_path, index=False, encoding='utf-8-sig')
        
        print(f"成功转换: {shp_file} -> {csv_filename}")
        
    except Exception as e:
        print(f"转换 {shp_file} 时出错: {str(e)}")

print("所有shp文件转换完成！")


In [ ]:
import os
import geopandas as gpd

# 设置文件夹路径和文件名
folder_path = r"E:\chuli\shpfile"
shp_filename = "HK_landuse.shp"
shp_path = os.path.join(folder_path, shp_filename)

# 检查文件是否存在
if not os.path.exists(shp_path):
    print(f"错误: 文件 {shp_path} 不存在!")
else:
    try:
        # 读取shp文件
        gdf = gpd.read_file(shp_path)
        
        print(f"成功读取文件: {shp_filename}")
        print(f"要素数量: {len(gdf)}")
        print(f"几何类型: {gdf.geometry.type.iloc[0] if len(gdf) > 0 else 'N/A'}")
        print(f"坐标参考系统: {gdf.crs}")
        print(f"字段信息: {list(gdf.columns)}")
        
        # 构建输出geojson文件名
        geojson_filename = os.path.splitext(shp_filename)[0] + '.geojson'
        geojson_path = os.path.join(folder_path, geojson_filename)
        
        # 保存为geojson文件
        gdf.to_file(geojson_path, driver='GeoJSON', encoding='utf-8')
        
        print(f"成功转换: {shp_filename} -> {geojson_filename}")
        print(f"输出路径: {geojson_path}")
        
    except Exception as e:
        print(f"转换过程中出错: {str(e)}")

print("转换完成！")


In [ ]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy import stats
from shapely.geometry import Point
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV

def calculate_density_analysis_with_raster():
    # 设置文件夹路径
    data_folder = r"E:\chuli\traffic"
    school_file = r"E:\chuli\over\school_NY.csv"
    
    # 读取学校数据，尝试不同的编码
    try:
        # 首先尝试gbk编码
        school_df = pd.read_csv(school_file, encoding='gbk')
        print(f"成功使用GBK编码读取学校数据，共 {len(school_df)} 条记录")
    except UnicodeDecodeError:
        try:
            # 如果gbk失败，尝试gb2312编码
            school_df = pd.read_csv(school_file, encoding='gb2312')
            print(f"成功使用GB2312编码读取学校数据，共 {len(school_df)} 条记录")
        except UnicodeDecodeError:
            try:
                # 如果gb2312也失败，尝试latin-1编码
                school_df = pd.read_csv(school_file, encoding='latin-1')
                print(f"成功使用Latin-1编码读取学校数据，共 {len(school_df)} 条记录")
            except UnicodeDecodeError:
                print("所有编码尝试都失败了，使用错误处理方式读取")
                school_df = pd.read_csv(school_file, encoding='utf-8', errors='ignore')
                print(f"使用错误忽略方式读取学校数据，共 {len(school_df)} 条记录")
    
    # 假设学校数据有经纬度列，如果没有请根据实际列名调整
    if 'longitude' in school_df.columns and 'latitude' in school_df.columns:
        school_points = school_df[['longitude', 'latitude']].dropna().values
    elif 'lng' in school_df.columns and 'lat' in school_df.columns:
        school_points = school_df[['lng', 'lat']].dropna().values
    elif 'x' in school_df.columns and 'y' in school_df.columns:
        school_points = school_df[['x', 'y']].dropna().values
    else:
        # 如果没有明确的经纬度列，尝试查找可能的列名
        possible_lon_cols = ['lon', 'longitude', 'lng', 'x', '经度', 'X']
        possible_lat_cols = ['lat', 'latitude', 'y', '纬度', 'Y']
        
        lon_col = None
        lat_col = None
        for col in school_df.columns:
            if any(keyword in col.lower() or keyword in col for keyword in possible_lon_cols):
                lon_col = col
            if any(keyword in col.lower() or keyword in col for keyword in possible_lat_cols):
                lat_col = col
        
        if lon_col and lat_col:
            school_points = school_df[[lon_col, lat_col]].dropna().values
            print(f"使用列 '{lon_col}' 和 '{lat_col}' 作为经纬度")
        else:
            print("错误: 无法找到经纬度列，请检查学校数据的列名")
            print(f"可用列名: {list(school_df.columns)}")
            return
    
    print(f"有效学校坐标点数量: {len(school_points)}")
    
    # 获取文件夹中所有以HK开头的文件
    files = [f for f in os.listdir(data_folder) if f.startswith('HK')]
    
    if not files:
        print("在指定文件夹中未找到以HK开头的文件")
        return
    
    print(f"找到 {len(files)} 个以HK开头的文件:")
    for file in files:
        print(f"  - {file}")
    
    # 处理每个文件
    for file in files:
        file_path = os.path.join(data_folder, file)
        
        try:
            # 根据文件扩展名选择读取方法
            if file.lower().endswith('.shp'):
                # 读取shp文件
                gdf = gpd.read_file(file_path)
                print(f"\n处理文件: {file}")
                print(f"要素数量: {len(gdf)}")
                print(f"几何类型: {gdf.geometry.type.iloc[0] if len(gdf) > 0 else 'N/A'}")
                
                # 提取坐标点
                if gdf.geometry.type.iloc[0] == 'Point':
                    points = np.column_stack((gdf.geometry.x, gdf.geometry.y))
                else:
                    # 对于面要素，使用中心点
                    centroids = gdf.geometry.centroid
                    points = np.column_stack((centroids.x, centroids.y))
                
            elif file.lower().endswith(('.csv', '.txt')):
                # 读取CSV文件，尝试不同的编码
                df = None
                encodings = ['utf-8', 'gbk', 'gb2312', 'latin-1']
                
                for encoding in encodings:
                    try:
                        df = pd.read_csv(file_path, encoding=encoding)
                        print(f"使用 {encoding} 编码成功读取 {file}")
                        break
                    except UnicodeDecodeError:
                        continue
                
                if df is None:
                    print(f"无法读取文件 {file}，所有编码尝试都失败了")
                    continue
                
                print(f"\n处理文件: {file}")
                print(f"记录数量: {len(df)}")
                
                # 查找经纬度列
                if 'longitude' in df.columns and 'latitude' in df.columns:
                    points = df[['longitude', 'latitude']].dropna().values
                elif 'lng' in df.columns and 'lat' in df.columns:
                    points = df[['lng', 'lat']].dropna().values
                elif 'x' in df.columns and 'y' in df.columns:
                    points = df[['x', 'y']].dropna().values
                else:
                    # 尝试其他可能的列名
                    possible_lon_cols = ['lon', 'longitude', 'lng', 'x', '经度', 'X']
                    possible_lat_cols = ['lat', 'latitude', 'y', '纬度', 'Y']
                    
                    lon_col = None
                    lat_col = None
                    for col in df.columns:
                        if any(keyword in col.lower() or keyword in col for keyword in possible_lon_cols):
                            lon_col = col
                        if any(keyword in col.lower() or keyword in col for keyword in possible_lat_cols):
                            lat_col = col
                    
                    if lon_col and lat_col:
                        points = df[[lon_col, lat_col]].dropna().values
                        print(f"使用列 '{lon_col}' 和 '{lat_col}' 作为经纬度")
                    else:
                        print(f"  错误: 无法在 {file} 中找到经纬度列")
                        print(f"  可用列名: {list(df.columns)}")
                        continue
            
            else:
                print(f"  跳过不支持的文件类型: {file}")
                continue
            
            if len(points) == 0:
                print(f"  错误: {file} 中没有有效的坐标数据")
                continue
            
            print(f"  有效坐标点数量: {len(points)}")
            
            # 计算边界框
            min_lon, max_lon = min(np.min(points[:, 0]), np.min(school_points[:, 0])), max(np.max(points[:, 0]), np.max(school_points[:, 0]))
            min_lat, max_lat = min(np.min(points[:, 1]), np.min(school_points[:, 1])), max(np.max(points[:, 1]), np.max(school_points[:, 1]))
            
            # 定义栅格参数（10m×10m像素）
            # 注意：经纬度坐标系下，10米对应的角度差约为 10/(111*1000) ≈ 0.00009 度
            pixel_size = 0.00009  # 约10米
            
            # 计算栅格尺寸
            width = int((max_lon - min_lon) / pixel_size)
            height = int((max_lat - min_lat) / pixel_size)
            
            # 确保最小尺寸
            width = max(width, 100)
            height = max(height, 100)
            
            print(f"  栅格尺寸: {width} x {height}")
            print(f"  像素大小: {pixel_size} 度 (约10米)")
            
            # 创建坐标网格
            lon_range = np.linspace(min_lon, max_lon, width)
            lat_range = np.linspace(min_lat, max_lat, height)
            lon_mesh, lat_mesh = np.meshgrid(lon_range, lat_range)
            grid_points = np.column_stack((lon_mesh.ravel(), lat_mesh.ravel()))
            
            # 使用核密度估计
            kde = KernelDensity(bandwidth=0.0001, kernel='gaussian')  # 带宽调整为约10米
            kde.fit(points)
            
            # 在网格点上评估密度
            log_dens = kde.score_samples(grid_points)
            dens = np.exp(log_dens)
            dens_grid = dens.reshape(height, width)
            
            # 计算每个学校位置的密度值
            school_log_dens = kde.score_samples(school_points)
            school_densities = np.exp(school_log_dens)
            
            # 计算面积（估算）
            area = (max_lon - min_lon) * (max_lat - min_lat) * 111 * 111  # 约111km/度
            area_km2 = area  # 平方公里
            
            # 计算数量与面积比
            count = len(points)
            density_ratio = count / area_km2 if area_km2 > 0 else 0
            
            print(f"  数据点数量: {count}")
            print(f"  估算面积: {area_km2:.2f} 平方公里")
            print(f"  数量与面积比: {density_ratio:.4f} 个/平方公里")
            print(f"  学校位置平均密度: {np.mean(school_densities):.6f}")
            print(f"  最大密度: {np.max(school_densities):.6f}")
            print(f"  最小密度: {np.min(school_densities):.6f}")
            
            # 保存核密度栅格数据为numpy数组格式
            raster_filename = os.path.join(data_folder, f"{os.path.splitext(file)[0]}_density_kernel.npy")
            np.save(raster_filename, dens_grid)
            print(f"  核密度栅格已保存到: {raster_filename}")
            
            # 保存为CSV格式的栅格数据（包含坐标信息）
            csv_raster_filename = os.path.join(data_folder, f"{os.path.splitext(file)[0]}_density_kernel.csv")
            
            # 创建坐标和密度值的数据框
            lon_coords, lat_coords = np.meshgrid(lon_range, lat_range)
            raster_df = pd.DataFrame({
                'longitude': lon_coords.ravel(),
                'latitude': lat_coords.ravel(),
                'density': dens_grid.ravel()
            })
            raster_df.to_csv(csv_raster_filename, index=False, encoding='utf-8-sig')
            print(f"  核密度栅格CSV已保存到: {csv_raster_filename}")
            
            # 保存分析结果
            output_file = os.path.join(data_folder, f"{os.path.splitext(file)[0]}_density_analysis.csv")
            result_df = pd.DataFrame({
                'filename': [file],
                'count': [count],
                'area_km2': [area_km2],
                'density_ratio': [density_ratio],
                'avg_school_density': [np.mean(school_densities)],
                'max_density': [np.max(school_densities)],
                'min_density': [np.min(school_densities)],
                'raster_file': [raster_filename],
                'csv_raster_file': [csv_raster_filename]
            })
            result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
            print(f"  分析结果已保存到: {output_file}")
            
        except Exception as e:
            print(f"处理文件 {file} 时出错: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    print("\n所有文件处理完成！")

# 运行分析
calculate_density_analysis_with_raster()


In [ ]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy import stats
from shapely.geometry import Point
import matplotlib.pyplot as plt
from sklearn.neighbors import KernelDensity
from sklearn.model_selection import GridSearchCV

def calculate_density_analysis_with_raster():
    # 设置文件夹路径
    data_folder = r"E:\chuli\traffic"
    school_file = r"E:\chuli\over\school_NY.csv"  # 修改为NY
    
    # 读取学校数据，尝试不同的编码
    try:
        # 首先尝试gbk编码
        school_df = pd.read_csv(school_file, encoding='gbk')
        print(f"成功使用GBK编码读取学校数据，共 {len(school_df)} 条记录")
    except UnicodeDecodeError:
        try:
            # 如果gbk失败，尝试gb2312编码
            school_df = pd.read_csv(school_file, encoding='gb2312')
            print(f"成功使用GB2312编码读取学校数据，共 {len(school_df)} 条记录")
        except UnicodeDecodeError:
            try:
                # 如果gb2312也失败，尝试latin-1编码
                school_df = pd.read_csv(school_file, encoding='latin-1')
                print(f"成功使用Latin-1编码读取学校数据，共 {len(school_df)} 条记录")
            except UnicodeDecodeError:
                print("所有编码尝试都失败了，使用错误处理方式读取")
                school_df = pd.read_csv(school_file, encoding='utf-8', errors='ignore')
                print(f"使用错误忽略方式读取学校数据，共 {len(school_df)} 条记录")
    
    # 学校数据经纬度列名
    if 'Latitude' in school_df.columns and 'Longitude' in school_df.columns:
        school_points = school_df[['Longitude', 'Latitude']].dropna().values
    elif 'longitude' in school_df.columns and 'latitude' in school_df.columns:
        school_points = school_df[['longitude', 'latitude']].dropna().values
    elif 'lng' in school_df.columns and 'lat' in school_df.columns:
        school_points = school_df[['lng', 'lat']].dropna().values
    elif 'x' in school_df.columns and 'y' in school_df.columns:
        school_points = school_df[['x', 'y']].dropna().values
    else:
        print("错误: 无法找到经纬度列，请检查学校数据的列名")
        print(f"可用列名: {list(school_df.columns)}")
        return
    
    print(f"有效学校坐标点数量: {len(school_points)}")
    
    # 获取文件夹中所有以NY开头的文件
    files = [f for f in os.listdir(data_folder) if f.startswith('NY')]  # 修改为NY
    
    if not files:
        print("在指定文件夹中未找到以NY开头的文件")
        return
    
    print(f"找到 {len(files)} 个以NY开头的文件:")
    for file in files:
        print(f"  - {file}")
    
    # 处理每个文件
    for file in files:
        file_path = os.path.join(data_folder, file)
        
        try:
            # 根据文件扩展名选择读取方法
            if file.lower().endswith('.shp'):
                # 读取shp文件
                gdf = gpd.read_file(file_path)
                print(f"\n处理文件: {file}")
                print(f"要素数量: {len(gdf)}")
                print(f"几何类型: {gdf.geometry.type.iloc[0] if len(gdf) > 0 else 'N/A'}")
                
                # 提取坐标点
                if gdf.geometry.type.iloc[0] == 'Point':
                    points = np.column_stack((gdf.geometry.x, gdf.geometry.y))
                else:
                    # 对于面要素，使用中心点
                    centroids = gdf.geometry.centroid
                    points = np.column_stack((centroids.x, centroids.y))
                
            elif file.lower().endswith(('.csv', '.txt')):
                # 读取CSV文件，尝试不同的编码
                df = None
                encodings = ['utf-8', 'gbk', 'gb2312', 'latin-1']
                
                for encoding in encodings:
                    try:
                        df = pd.read_csv(file_path, encoding=encoding)
                        print(f"使用 {encoding} 编码成功读取 {file}")
                        break
                    except UnicodeDecodeError:
                        continue
                
                if df is None:
                    print(f"无法读取文件 {file}，所有编码尝试都失败了")
                    continue
                
                print(f"\n处理文件: {file}")
                print(f"记录数量: {len(df)}")
                
                # 学校数据经纬度列名
                if 'Latitude' in df.columns and 'Longitude' in df.columns:
                    points = df[['Longitude', 'Latitude']].dropna().values
                elif 'longitude' in df.columns and 'latitude' in df.columns:
                    points = df[['longitude', 'latitude']].dropna().values
                elif 'lng' in df.columns and 'lat' in df.columns:
                    points = df[['lng', 'lat']].dropna().values
                elif 'x' in df.columns and 'y' in df.columns:
                    points = df[['x', 'y']].dropna().values
                else:
                    print(f"  错误: 无法在 {file} 中找到经纬度列")
                    print(f"  可用列名: {list(df.columns)}")
                    continue
            
            else:
                print(f"  跳过不支持的文件类型: {file}")
                continue
            
            if len(points) == 0:
                print(f"  错误: {file} 中没有有效的坐标数据")
                continue
            
            print(f"  有效坐标点数量: {len(points)}")
            
            # 计算边界框
            min_lon, max_lon = min(np.min(points[:, 0]), np.min(school_points[:, 0])), max(np.max(points[:, 0]), np.max(school_points[:, 0]))
            min_lat, max_lat = min(np.min(points[:, 1]), np.min(school_points[:, 1])), max(np.max(points[:, 1]), np.max(school_points[:, 1]))
            
            # 定义栅格参数（50m×50m像素）
            # 注意：经纬度坐标系下，50米对应的角度差约为 50/(111*1000) ≈ 0.00045 度
            pixel_size = 0.00045  # 约50米
            
            # 计算栅格尺寸
            width = int((max_lon - min_lon) / pixel_size)
            height = int((max_lat - min_lat) / pixel_size)
            
            # 确保最小尺寸
            width = max(width, 20)
            height = max(height, 20)
            
            print(f"  栅格尺寸: {width} x {height}")
            print(f"  像素大小: {pixel_size} 度 (约50米)")
            
            # 创建坐标网格
            lon_range = np.linspace(min_lon, max_lon, width)
            lat_range = np.linspace(min_lat, max_lat, height)
            lon_mesh, lat_mesh = np.meshgrid(lon_range, lat_range)
            grid_points = np.column_stack((lon_mesh.ravel(), lat_mesh.ravel()))
            
            # 使用核密度估计
            kde = KernelDensity(bandwidth=0.0005, kernel='gaussian')  # 带宽调整为约50米
            kde.fit(points)
            
            # 在网格点上评估密度
            log_dens = kde.score_samples(grid_points)
            dens = np.exp(log_dens)
            dens_grid = dens.reshape(height, width)
            
            # 计算每个学校位置的密度值
            school_log_dens = kde.score_samples(school_points)
            school_densities = np.exp(school_log_dens)
            
            # 计算面积（估算）
            area = (max_lon - min_lon) * (max_lat - min_lat) * 111 * 111  # 约111km/度
            area_km2 = area  # 平方公里
            
            # 计算数量与面积比
            count = len(points)
            density_ratio = count / area_km2 if area_km2 > 0 else 0
            
            print(f"  数据点数量: {count}")
            print(f"  估算面积: {area_km2:.2f} 平方公里")
            print(f"  数量与面积比: {density_ratio:.4f} 个/平方公里")
            print(f"  学校位置平均密度: {np.mean(school_densities):.6f}")
            print(f"  最大密度: {np.max(school_densities):.6f}")
            print(f"  最小密度: {np.min(school_densities):.6f}")
            
            # 保存核密度栅格数据为numpy数组格式
            raster_filename = os.path.join(data_folder, f"{os.path.splitext(file)[0]}_density_kernel.npy")
            np.save(raster_filename, dens_grid)
            print(f"  核密度栅格已保存到: {raster_filename}")
            
            # 保存为CSV格式的栅格数据（包含坐标信息）
            csv_raster_filename = os.path.join(data_folder, f"{os.path.splitext(file)[0]}_density_kernel.csv")
            
            # 创建坐标和密度值的数据框
            lon_coords, lat_coords = np.meshgrid(lon_range, lat_range)
            raster_df = pd.DataFrame({
                'Longitude': lon_coords.ravel(),
                'Latitude': lat_coords.ravel(),
                'density': dens_grid.ravel()
            })
            raster_df.to_csv(csv_raster_filename, index=False, encoding='utf-8-sig')
            print(f"  核密度栅格CSV已保存到: {csv_raster_filename}")
            
            # 保存分析结果
            output_file = os.path.join(data_folder, f"{os.path.splitext(file)[0]}_density_analysis.csv")
            result_df = pd.DataFrame({
                'filename': [file],
                'count': [count],
                'area_km2': [area_km2],
                'density_ratio': [density_ratio],
                'avg_school_density': [np.mean(school_densities)],
                'max_density': [np.max(school_densities)],
                'min_density': [np.min(school_densities)],
                'raster_file': [raster_filename],
                'csv_raster_file': [csv_raster_filename]
            })
            result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
            print(f"  分析结果已保存到: {output_file}")
            
        except Exception as e:
            print(f"处理文件 {file} 时出错: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    print("\n所有文件处理完成！")

# 运行分析
calculate_density_analysis_with_raster()
